# AgeLens — 08 Canonical V1 Output Rebuild

This notebook rebuilds the governed **AgeLens V1 canonical outputs** after the governance resolution completed in notebook 07.

## Canonical specification

- Primary Phenotypic Age variant: **Supplement**
- Final conversion pair: `141.50225 / 0.090165`
- Canonical creatinine scale: observed modern harmonized creatinine
- Age top-coding: retain `RIDAGEYR == 80`, set `age_topcoded = TRUE`
- Missing data: complete-case analysis
- Survey weight: pooled fasting subsample weight `WTSAF4YR`
- Canonical cycles: 2015–2016 and 2017–2018

## Required sensitivities

- Erratum final conversion pair
- No-topcode sample
- Creatinine shifts of `+0.11`, `+0.17`, and `+0.23 mg/dL`

## Release behavior

The notebook keeps the release gate closed until every canonical regression check passes. On success:

- cross-sectional AgeLens V1 outputs become reportable;
- mortality analysis remains unauthorized;
- no mortality data are read;
- the configuration update is backed up and audited.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import shutil

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")


numpy: 2.4.6
pandas: 2.3.3
Current working directory: <PROJECT_ROOT>\notebooks


In [2]:
def find_project_root(
    folder_name: str = PROJECT_FOLDER_NAME,
) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'."
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


PROJECT_ROOT = find_project_root()
CONFIG_PATH = (
    PROJECT_ROOT / "configs" / "agelens_config.json"
)

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration not found: {CONFIG_PATH}"
    )

CONFIG = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

INTERIM_ROOT = (
    PROJECT_ROOT / CONFIG["paths"]["interim_data"]
)
PROCESSED_ROOT = (
    PROJECT_ROOT / CONFIG["paths"]["processed_data"]
)
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]
REPORT_ROOT = (
    PROJECT_ROOT / "docs" / "methodology"
)

for path in [
    PROCESSED_ROOT,
    TABLES_ROOT,
    LOGS_ROOT,
    REPORT_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

PREPROCESSED_PATH = (
    INTERIM_ROOT
    / "nhanes_2015_2018_preprocessed_diagnostic.parquet"
)
EG004_PARTICIPANT_PATH = (
    PROCESSED_ROOT
    / "06_eg004_creatinine_sensitivity_diagnostic.parquet"
)
VALIDATION_SUMMARY_PATH = (
    TABLES_ROOT / "03_survey_weighted_summaries.csv"
)
VALIDATION_SAMPLE_PATH = (
    TABLES_ROOT / "03_validation_sample_counts.csv"
)
BIOAGE_COMPARISON_PATH = (
    TABLES_ROOT / "03_bioage_comparison.csv"
)
VALIDATION_STATUS_PATH = (
    TABLES_ROOT / "05_validation_check_status.csv"
)
EG004_SUMMARY_PATH = (
    TABLES_ROOT
    / "06_eg004_creatinine_sensitivity_summary.csv"
)
EG004_REFERENCE_PATH = (
    TABLES_ROOT
    / "06_eg004_reference_equivalence_check.csv"
)
GOVERNANCE_AUDIT_PATH = (
    TABLES_ROOT
    / "07_governance_consistency_checks.csv"
)

required_inputs = [
    PREPROCESSED_PATH,
    EG004_PARTICIPANT_PATH,
    VALIDATION_SUMMARY_PATH,
    VALIDATION_SAMPLE_PATH,
    BIOAGE_COMPARISON_PATH,
    VALIDATION_STATUS_PATH,
    EG004_SUMMARY_PATH,
    EG004_REFERENCE_PATH,
    GOVERNANCE_AUDIT_PATH,
]

missing_inputs = [
    path
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        f"Required canonical rebuild inputs are missing: {missing_inputs}"
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Preprocessed input: {PREPROCESSED_PATH}")
print(f"EG-004 input: {EG004_PARTICIPANT_PATH}")


Project root: <PROJECT_ROOT>
Preprocessed input: <PROJECT_ROOT>\data\interim\nhanes_2015_2018_preprocessed_diagnostic.parquet
EG-004 input: <PROJECT_ROOT>\data\processed\06_eg004_creatinine_sensitivity_diagnostic.parquet


## 1. Verify governance and release prerequisites


In [3]:
expected_decisions = {
    "formula_constants": "D-010",
    "age_topcoding": "D-011",
    "creatinine_training_scale": "D-012",
    "validation_acceptance": "D-013",
    "xpt_zero_normalization": "D-014",
}

actual_decisions = CONFIG["governance"].get(
    "decisions",
    {},
)

for key, expected_decision in expected_decisions.items():
    actual_decision = actual_decisions.get(key)

    if actual_decision != expected_decision:
        raise RuntimeError(
            f"Governance decision {key} must be "
            f"{expected_decision}; found {actual_decision!r}."
        )

if CONFIG["governance"].get(
    "open_core_evidence_gaps"
) != []:
    raise RuntimeError(
        "Open Core Evidence Gaps remain in the configuration."
    )

if CONFIG["formula"].get(
    "primary_variant"
) != "supplement":
    raise RuntimeError(
        "The configured primary formula variant is not Supplement."
    )

release_gates = CONFIG.get(
    "release_gates",
    {},
)

if not release_gates.get(
    "governance_core_gaps_resolved",
    False,
):
    raise RuntimeError(
        "The governance Core-gap release gate is not resolved."
    )

if not release_gates.get(
    "validation_completed",
    False,
):
    raise RuntimeError(
        "The validation-completed release gate is not resolved."
    )

if release_gates.get(
    "mortality_analysis_authorized",
    False,
):
    raise RuntimeError(
        "Mortality analysis is unexpectedly authorized. "
        "Notebook 08 must not operate under that state."
    )

governance_checks = pd.read_csv(
    GOVERNANCE_AUDIT_PATH
)

if not governance_checks["pass"].astype(bool).all():
    raise RuntimeError(
        "Notebook 07 governance consistency checks did not all pass."
    )

validation_status = pd.read_csv(
    VALIDATION_STATUS_PATH
)

required_statuses = {
    "Check 1 — Age correlation vs BioAge": "PASS",
    "Check 2 — Cross-implementation agreement": (
        "PASS_BASELINE_ESTABLISHED"
    ),
    "Check 3 — Bridging effectiveness": "PASS",
}

status_lookup = dict(
    zip(
        validation_status["check"],
        validation_status["status"],
    )
)

for check_name, expected_status in required_statuses.items():
    if status_lookup.get(check_name) != expected_status:
        raise RuntimeError(
            f"{check_name} does not satisfy the canonical "
            f"rebuild prerequisite: {status_lookup.get(check_name)!r}"
        )

print("✅ Governance and release prerequisites verified.")


RuntimeError: Mortality analysis is unexpectedly authorized. Notebook 08 must not operate under that state.

## 2. Build the canonical participant-level sample


In [ ]:
preprocessed = pd.read_parquet(
    PREPROCESSED_PATH
)
eg004 = pd.read_parquet(
    EG004_PARTICIPANT_PATH
)

required_preprocessed_columns = {
    "SEQN",
    "NHANES_CYCLE",
    "chronological_age_years",
    "age_topcoded",
    "age_below_20",
    "WTSAF4YR",
    "SDMVSTRA",
    "SDMVPSU",
    "complete_case_harmonized",
    "harmonized_phenoage_supplement_years",
    "harmonized_phenoage_erratum_years",
}

missing_preprocessed_columns = sorted(
    required_preprocessed_columns
    - set(preprocessed.columns)
)

if missing_preprocessed_columns:
    raise ValueError(
        "Preprocessed input is missing required columns: "
        f"{missing_preprocessed_columns}"
    )

required_eg004_columns = {
    "SEQN",
    "NHANES_CYCLE",
    "eg004_nhanes3_bias_low_phenoage_supplement_years",
    "eg004_nhanes3_bias_mid_phenoage_supplement_years",
    "eg004_nhanes3_bias_high_phenoage_supplement_years",
    "eg004_nhanes3_bias_low_delta_supplement_years",
    "eg004_nhanes3_bias_mid_delta_supplement_years",
    "eg004_nhanes3_bias_high_delta_supplement_years",
}

missing_eg004_columns = sorted(
    required_eg004_columns
    - set(eg004.columns)
)

if missing_eg004_columns:
    raise ValueError(
        "EG-004 input is missing required columns: "
        f"{missing_eg004_columns}"
    )

canonical = preprocessed.loc[
    preprocessed["complete_case_harmonized"]
    & preprocessed["WTSAF4YR"].notna()
    & preprocessed["WTSAF4YR"].gt(0)
].copy()

if canonical.empty:
    raise RuntimeError(
        "The canonical complete-case sample is empty."
    )

if canonical.duplicated(
    ["NHANES_CYCLE", "SEQN"]
).any():
    raise RuntimeError(
        "Duplicate cycle + SEQN rows exist in the canonical sample."
    )

eg004_merge_columns = sorted(
    required_eg004_columns
)

canonical = canonical.merge(
    eg004.loc[:, eg004_merge_columns],
    on=["SEQN", "NHANES_CYCLE"],
    how="left",
    validate="one_to_one",
)

if canonical[
    list(
        required_eg004_columns
        - {"SEQN", "NHANES_CYCLE"}
    )
].isna().any().any():
    raise RuntimeError(
        "One or more canonical participants lack EG-004 sensitivity values."
    )

canonical["phenotypic_age_years"] = (
    canonical[
        "harmonized_phenoage_supplement_years"
    ]
)
canonical[
    "phenotypic_age_minus_chronological_age_years"
] = (
    canonical["phenotypic_age_years"]
    - canonical["chronological_age_years"]
)

canonical["formula_variant"] = "supplement"
canonical["formula_intercept"] = 141.50225
canonical["formula_denominator"] = 0.090165
canonical["creatinine_scale"] = (
    "observed_modern_harmonized"
)
canonical["diagnostic_only"] = False
canonical["canonical_v1"] = True
canonical["cross_sectional_reportable"] = True
canonical["mortality_analysis_authorized"] = False

canonical[
    "sensitivity_erratum_phenoage_years"
] = canonical[
    "harmonized_phenoage_erratum_years"
]

canonical[
    "sensitivity_creatinine_plus_0_11_phenoage_years"
] = canonical[
    "eg004_nhanes3_bias_low_phenoage_supplement_years"
]
canonical[
    "sensitivity_creatinine_plus_0_17_phenoage_years"
] = canonical[
    "eg004_nhanes3_bias_mid_phenoage_supplement_years"
]
canonical[
    "sensitivity_creatinine_plus_0_23_phenoage_years"
] = canonical[
    "eg004_nhanes3_bias_high_phenoage_supplement_years"
]

canonical["pooled_stratum"] = (
    canonical["NHANES_CYCLE"].astype(str)
    + "__"
    + canonical["SDMVSTRA"].astype(str)
)
canonical["pooled_psu"] = (
    canonical["NHANES_CYCLE"].astype(str)
    + "__"
    + canonical["SDMVSTRA"].astype(str)
    + "__"
    + canonical["SDMVPSU"].astype(str)
)

finite_columns = [
    "phenotypic_age_years",
    "sensitivity_erratum_phenoage_years",
    "sensitivity_creatinine_plus_0_11_phenoage_years",
    "sensitivity_creatinine_plus_0_17_phenoage_years",
    "sensitivity_creatinine_plus_0_23_phenoage_years",
]

if not np.isfinite(
    canonical[finite_columns].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Canonical or required sensitivity outputs contain "
        "non-finite values."
    )

sample_flow = (
    canonical.groupby(
        "NHANES_CYCLE",
        observed=True,
    )
    .agg(
        n=("SEQN", "size"),
        weighted_population_sum=(
            "WTSAF4YR",
            "sum",
        ),
        age_topcoded_n=(
            "age_topcoded",
            "sum",
        ),
        age_below_20_n=(
            "age_below_20",
            "sum",
        ),
    )
    .reset_index()
)

sample_flow["no_topcode_n"] = sample_flow[
    "NHANES_CYCLE"
].map(
    canonical.loc[
        ~canonical["age_topcoded"]
    ]
    .groupby(
        "NHANES_CYCLE",
        observed=True,
    )
    .size()
)

display(sample_flow)
print(f"Canonical participant rows: {len(canonical):,}")


,NHANES_CYCLE,n,weighted_population_sum,age_topcoded_n,age_below_20_n,no_topcode_n
0,2015_2016,2645,1.292913e+08,121,464,2524
1,2017_2018,2578,1.302142e+08,151,392,2427


Canonical participant rows: 5,223


## 3. Calculate survey-weighted canonical summaries


In [ ]:
def survey_mean_taylor(
    frame: pd.DataFrame,
    value: str,
    *,
    weight: str = "WTSAF4YR",
    stratum: str = "SDMVSTRA",
    psu: str = "SDMVPSU",
) -> dict[str, float]:
    valid = frame.loc[
        :,
        [value, weight, stratum, psu],
    ].dropna().copy()

    valid = valid.loc[
        valid[weight].gt(0)
    ]

    if valid.empty:
        raise RuntimeError(
            f"No valid observations for {value}."
        )

    estimate = float(
        np.average(
            valid[value],
            weights=valid[weight],
        )
    )
    total_weight = float(
        valid[weight].sum()
    )

    valid["_linearized"] = (
        valid[weight]
        * (valid[value] - estimate)
        / total_weight
    )

    psu_totals = (
        valid.groupby(
            [stratum, psu],
            observed=True,
        )["_linearized"]
        .sum()
        .reset_index()
    )

    variance = 0.0

    for stratum_value, stratum_frame in psu_totals.groupby(
        stratum,
        observed=True,
    ):
        psu_values = stratum_frame[
            "_linearized"
        ].to_numpy(dtype=float)
        psu_count = len(psu_values)

        if psu_count < 2:
            raise RuntimeError(
                "A stratum has fewer than two observed PSUs: "
                f"{stratum_value}"
            )

        centered = (
            psu_values - psu_values.mean()
        )
        variance += (
            psu_count
            / (psu_count - 1)
            * np.square(centered).sum()
        )

    standard_error = float(
        np.sqrt(variance)
    )

    weighted_variance = float(
        np.average(
            np.square(
                valid[value] - estimate
            ),
            weights=valid[weight],
        )
    )

    return {
        "n": int(len(valid)),
        "weighted_population_sum": total_weight,
        "weighted_mean": estimate,
        "taylor_se": standard_error,
        "ci_low_95": (
            estimate - 1.96 * standard_error
        ),
        "ci_high_95": (
            estimate + 1.96 * standard_error
        ),
        "weighted_sd_descriptive": float(
            np.sqrt(weighted_variance)
        ),
        "stratum_count": int(
            valid[stratum].nunique()
        ),
        "psu_count": int(
            valid[
                [stratum, psu]
            ].drop_duplicates().shape[0]
        ),
    }


summary_records = []

sample_definitions = {
    "canonical_full": pd.Series(
        True,
        index=canonical.index,
    ),
    "sensitivity_no_topcode": (
        ~canonical["age_topcoded"]
    ),
}

measure_definitions = {
    "canonical_supplement": (
        "phenotypic_age_years"
    ),
    "sensitivity_erratum": (
        "sensitivity_erratum_phenoage_years"
    ),
    "sensitivity_creatinine_plus_0_11": (
        "sensitivity_creatinine_plus_0_11_phenoage_years"
    ),
    "sensitivity_creatinine_plus_0_17": (
        "sensitivity_creatinine_plus_0_17_phenoage_years"
    ),
    "sensitivity_creatinine_plus_0_23": (
        "sensitivity_creatinine_plus_0_23_phenoage_years"
    ),
}

for sample_name, sample_mask in sample_definitions.items():
    sample_frame = canonical.loc[
        sample_mask
    ]

    for cycle, cycle_frame in sample_frame.groupby(
        "NHANES_CYCLE",
        observed=True,
    ):
        for measure_name, value_column in measure_definitions.items():
            survey = survey_mean_taylor(
                cycle_frame,
                value_column,
            )

            summary_records.append(
                {
                    "sample": sample_name,
                    "cycle": cycle,
                    "measure": measure_name,
                    "value_column": value_column,
                    **survey,
                }
            )

canonical_summary = pd.DataFrame(
    summary_records
)

display(
    canonical_summary.loc[
        canonical_summary["measure"].eq(
            "canonical_supplement"
        )
    ].round(6)
)


,sample,cycle,measure,value_column,n,weighted_population_sum,weighted_mean,taylor_se,ci_low_95,ci_high_95,weighted_sd_descriptive,stratum_count,psu_count
0,canonical_full,2015_2016,canonical_supplement,phenotypic_age_years,2645,1.292913e+08,42.140869,0.830376,40.513332,43.768406,21.439843,15,30
5,canonical_full,2017_2018,canonical_supplement,phenotypic_age_years,2578,1.302142e+08,43.225918,0.660650,41.931044,44.520791,21.859664,15,30
10,sensitivity_no_topcode,2015_2016,canonical_supplement,phenotypic_age_years,2524,1.251893e+08,40.806280,0.780352,39.276791,42.335770,20.389702,15,30
15,sensitivity_no_topcode,2017_2018,canonical_supplement,phenotypic_age_years,2427,1.252206e+08,41.615735,0.584285,40.470535,42.760934,20.663278,15,30


## 4. Run canonical regression checks


The mortality safeguard distinguishes NCHS linked-mortality fields from the Levine formula's internal `mortality_score` calculation. Formula intermediates are allowed; linked outcome fields are prohibited.


In [ ]:
validation_summary = pd.read_csv(
    VALIDATION_SUMMARY_PATH
)
validation_samples = pd.read_csv(
    VALIDATION_SAMPLE_PATH
)
bioage_comparison = pd.read_csv(
    BIOAGE_COMPARISON_PATH
)
eg004_summary = pd.read_csv(
    EG004_SUMMARY_PATH
)
eg004_reference = pd.read_csv(
    EG004_REFERENCE_PATH
)

check_records = []


def record_check(
    check: str,
    passed: bool,
    observed: object,
    expected: object,
    tolerance: object = None,
) -> None:
    check_records.append(
        {
            "check": check,
            "pass": bool(passed),
            "observed": observed,
            "expected": expected,
            "tolerance": tolerance,
        }
    )


# Check A: participant-level canonical values reproduce the
# governed Supplement diagnostic column exactly.
max_reference_difference = float(
    np.max(
        np.abs(
            canonical[
                "phenotypic_age_years"
            ].to_numpy(dtype=float)
            - canonical[
                "harmonized_phenoage_supplement_years"
            ].to_numpy(dtype=float)
        )
    )
)

record_check(
    "Canonical participant values reproduce Supplement reference",
    max_reference_difference <= 1e-12,
    max_reference_difference,
    0.0,
    1e-12,
)


# Check B: sample counts match notebook 03.
expected_sample_names = {
    "canonical_full": (
        "all_harmonized_complete_case"
    ),
    "sensitivity_no_topcode": "no_topcode",
}

for canonical_sample, validation_sample in expected_sample_names.items():
    observed_rows = canonical_summary.loc[
        canonical_summary["sample"].eq(
            canonical_sample
        )
        & canonical_summary["measure"].eq(
            "canonical_supplement"
        ),
        ["cycle", "n"],
    ].sort_values("cycle")

    expected_rows = validation_samples.loc[
        validation_samples["sample"].eq(
            validation_sample
        ),
        ["cycle", "n"],
    ].sort_values("cycle")

    merged_counts = observed_rows.merge(
        expected_rows,
        on="cycle",
        suffixes=("_observed", "_expected"),
        validate="one_to_one",
    )

    count_match = (
        merged_counts["n_observed"]
        .eq(merged_counts["n_expected"])
        .all()
    )

    record_check(
        f"{canonical_sample} counts match notebook 03",
        count_match,
        merged_counts["n_observed"].tolist(),
        merged_counts["n_expected"].tolist(),
        0,
    )


# Check C: survey means and SEs match notebook 03 Supplement rows.
for canonical_sample, validation_sample in expected_sample_names.items():
    observed_rows = canonical_summary.loc[
        canonical_summary["sample"].eq(
            canonical_sample
        )
        & canonical_summary["measure"].eq(
            "canonical_supplement"
        ),
        [
            "cycle",
            "weighted_mean",
            "taylor_se",
        ],
    ]

    expected_rows = validation_summary.loc[
        validation_summary["sample"].eq(
            validation_sample
        )
        & validation_summary[
            "formula_variant"
        ].eq("supplement"),
        [
            "cycle",
            "weighted_mean",
            "taylor_se",
        ],
    ]

    comparison = observed_rows.merge(
        expected_rows,
        on="cycle",
        suffixes=("_observed", "_expected"),
        validate="one_to_one",
    )

    maximum_mean_difference = float(
        np.max(
            np.abs(
                comparison[
                    "weighted_mean_observed"
                ]
                - comparison[
                    "weighted_mean_expected"
                ]
            )
        )
    )
    maximum_se_difference = float(
        np.max(
            np.abs(
                comparison[
                    "taylor_se_observed"
                ]
                - comparison[
                    "taylor_se_expected"
                ]
            )
        )
    )

    record_check(
        f"{canonical_sample} weighted means match notebook 03",
        maximum_mean_difference <= 1e-10,
        maximum_mean_difference,
        0.0,
        1e-10,
    )
    record_check(
        f"{canonical_sample} Taylor SEs match notebook 03",
        maximum_se_difference <= 1e-10,
        maximum_se_difference,
        0.0,
        1e-10,
    )


# Check D: BioAge first-run baseline matches configuration.
configured_baseline = CONFIG[
    "validation"
][
    "supplement_baseline_mae_by_cycle"
]

supplement_bioage = bioage_comparison.loc[
    bioage_comparison[
        "agelens_variant"
    ].eq("supplement")
].copy()

for _, row in supplement_bioage.iterrows():
    cycle = row["cycle"]
    observed_mae = float(row["mae"])
    expected_mae = float(
        configured_baseline[cycle]
    )

    record_check(
        f"BioAge Supplement MAE baseline matches for {cycle}",
        abs(observed_mae - expected_mae) <= 1e-12,
        observed_mae,
        expected_mae,
        1e-12,
    )

    record_check(
        f"BioAge Supplement MAE passes D-013 for {cycle}",
        observed_mae
        <= CONFIG["validation"][
            "check_2_max_mae_years"
        ],
        observed_mae,
        CONFIG["validation"][
            "check_2_max_mae_years"
        ],
        "<=",
    )

    record_check(
        f"BioAge Pearson passes D-013 for {cycle}",
        float(row["pearson"])
        >= CONFIG["validation"][
            "check_2_min_pearson"
        ],
        float(row["pearson"]),
        CONFIG["validation"][
            "check_2_min_pearson"
        ],
        ">=",
    )

    record_check(
        f"BioAge Spearman passes D-013 for {cycle}",
        float(row["spearman"])
        >= CONFIG["validation"][
            "check_2_min_spearman"
        ],
        float(row["spearman"]),
        CONFIG["validation"][
            "check_2_min_spearman"
        ],
        ">=",
    )


# Check E: EG-004 zero-shift equivalence and governed shifts.
record_check(
    "EG-004 zero-shift reference equivalence",
    eg004_reference["pass"].astype(bool).all(),
    bool(
        eg004_reference[
            "pass"
        ].astype(bool).all()
    ),
    True,
)

configured_shifts = CONFIG[
    "creatinine_training_scale_policy"
][
    "mandatory_sensitivity_shifts_mg_dL"
]

observed_shifts = sorted(
    eg004_summary.loc[
        eg004_summary["sample"].eq(
            "all_harmonized_complete_case"
        )
        & eg004_summary["domain"].eq(
            "pooled"
        )
        & eg004_summary[
            "formula_variant"
        ].eq("supplement")
        & eg004_summary["scenario"].ne(
            "observed_modern"
        ),
        "creatinine_shift_mg_dL",
    ]
    .drop_duplicates()
    .tolist()
)

record_check(
    "EG-004 governed sensitivity shifts are present",
    np.allclose(
        observed_shifts,
        configured_shifts,
        atol=0,
        rtol=0,
    ),
    observed_shifts,
    configured_shifts,
    0,
)


# Check F: required sensitivity behavior.
erratum_delta = (
    canonical[
        "sensitivity_erratum_phenoage_years"
    ]
    - canonical[
        "phenotypic_age_years"
    ]
)

record_check(
    "Erratum sensitivity differs from canonical Supplement",
    bool(
        np.max(
            np.abs(
                erratum_delta.to_numpy(dtype=float)
            )
        )
        > 1.0
    ),
    float(
        np.mean(
            erratum_delta.to_numpy(dtype=float)
        )
    ),
    "non-zero named sensitivity",
)

creatinine_delta_columns = [
    (
        "sensitivity_creatinine_plus_0_11_phenoage_years",
        0.11,
    ),
    (
        "sensitivity_creatinine_plus_0_17_phenoage_years",
        0.17,
    ),
    (
        "sensitivity_creatinine_plus_0_23_phenoage_years",
        0.23,
    ),
]

previous_mean_delta = -np.inf

for value_column, shift in creatinine_delta_columns:
    delta = (
        canonical[value_column]
        - canonical["phenotypic_age_years"]
    )
    mean_delta = float(
        delta.mean()
    )
    sd_delta = float(
        delta.std(ddof=0)
    )

    record_check(
        f"Creatinine +{shift:.2f} mg/dL sensitivity is positive",
        mean_delta > 0,
        mean_delta,
        "> 0",
    )
    record_check(
        f"Creatinine +{shift:.2f} mg/dL sensitivity is deterministic",
        sd_delta <= 1e-10,
        sd_delta,
        0.0,
        1e-10,
    )
    record_check(
        f"Creatinine +{shift:.2f} mg/dL sensitivity is monotonic",
        mean_delta > previous_mean_delta,
        mean_delta,
        f"> {previous_mean_delta}",
    )

    previous_mean_delta = mean_delta


# Check G: no linked-mortality fields.
#
# The Phenotypic Age formula legitimately contains intermediate
# variables such as `harmonized_mortality_score`. Those are
# mathematical formula outputs, not NCHS linked-mortality data.
# Therefore, inspect only known linked-mortality field names and
# prefixes rather than rejecting every column containing the word
# "mortality".
linked_mortality_exact_names = {
    "ELIGSTAT",
    "MORTSTAT",
    "UCOD_LEADING",
    "DIABETES",
    "HYPERTEN",
    "PERMTH_INT",
    "PERMTH_EXM",
    "ASSUMED_ALIVE",
    "ASSUMED_DECEASED",
    "LINKAGE_INELIGIBLE",
}

linked_mortality_prefixes = (
    "MORT_",
    "UCOD_",
    "PERMTH_",
    "LINKAGE_",
    "NCHS_MORTALITY_",
)

linked_mortality_columns = [
    column
    for column in canonical.columns
    if (
        column.upper() in linked_mortality_exact_names
        or column.upper().startswith(
            linked_mortality_prefixes
        )
    )
]

formula_mortality_score_columns = [
    column
    for column in canonical.columns
    if "mortality_score" in column.lower()
]

record_check(
    "Canonical source contains no linked-mortality fields",
    len(linked_mortality_columns) == 0,
    linked_mortality_columns,
    [],
)

record_check(
    "Formula mortality-score intermediates are classified separately",
    all(
        column not in linked_mortality_columns
        for column in formula_mortality_score_columns
    ),
    formula_mortality_score_columns,
    "Allowed formula intermediates; not linked mortality data",
)

regression_checks = pd.DataFrame(
    check_records
)

if not regression_checks["pass"].all():
    failed_checks = regression_checks.loc[
        ~regression_checks["pass"]
    ].copy()

    failure_audit_path = (
        TABLES_ROOT
        / "08_canonical_regression_failures.csv"
    )
    failed_checks.to_csv(
        failure_audit_path,
        index=False,
    )

    display(failed_checks)
    print(
        "Regression failure audit written: "
        f"{failure_audit_path}"
    )

    failure_names = "; ".join(
        failed_checks["check"].astype(str).tolist()
    )

    raise RuntimeError(
        "Canonical regression checks failed: "
        f"{failure_names}. "
        "No canonical release files or configuration updates "
        "were written."
    )

display(regression_checks)
print(
    f"✅ Canonical regression checks passed: "
    f"{len(regression_checks)}/{len(regression_checks)}"
)


,check,pass,observed,expected,tolerance
0,Canonical participant values reproduce Supplem...,True,0.0,0.0,0.0
1,canonical_full counts match notebook 03,True,"[2645, 2578]","[2645, 2578]",0
2,sensitivity_no_topcode counts match notebook 03,True,"[2524, 2427]","[2524, 2427]",0
3,canonical_full weighted means match notebook 03,True,0.0,0.0,0.0
4,canonical_full Taylor SEs match notebook 03,True,0.0,0.0,0.0
5,sensitivity_no_topcode weighted means match no...,True,0.0,0.0,0.0
6,sensitivity_no_topcode Taylor SEs match notebo...,True,0.0,0.0,0.0
7,BioAge Supplement MAE baseline matches for 201...,True,0.049726,0.049726,0.0
8,BioAge Supplement MAE passes D-013 for 2015_2016,True,0.049726,0.1,<=
9,BioAge Pearson passes D-013 for 2015_2016,True,1.0,0.999999,>=


✅ Canonical regression checks passed: 29/29


## 5. Write canonical datasets, sensitivity outputs, and reports


In [ ]:
CANONICAL_PARTICIPANT_PATH = (
    PROCESSED_ROOT
    / "agelens_v1_canonical_complete_case.parquet"
)
NO_TOPCODE_PARTICIPANT_PATH = (
    PROCESSED_ROOT
    / "agelens_v1_sensitivity_no_topcode.parquet"
)
SENSITIVITY_LONG_PATH = (
    PROCESSED_ROOT
    / "agelens_v1_required_sensitivities_long.parquet"
)
SAMPLE_FLOW_PATH = (
    TABLES_ROOT
    / "08_canonical_sample_flow.csv"
)
SUMMARY_PATH = (
    TABLES_ROOT
    / "08_canonical_survey_summary.csv"
)
REGRESSION_PATH = (
    TABLES_ROOT
    / "08_canonical_regression_checks.csv"
)
SENSITIVITY_SUMMARY_PATH = (
    TABLES_ROOT
    / "08_required_sensitivity_summary.csv"
)
MANIFEST_PATH = (
    TABLES_ROOT
    / "08_canonical_output_manifest.csv"
)
REPORT_PATH = (
    REPORT_ROOT
    / "Canonical_V1_Rebuild_Report.md"
)
METADATA_PATH = (
    LOGS_ROOT
    / "08_canonical_output_rebuild_metadata.json"
)

canonical_output_columns = [
    "SEQN",
    "NHANES_CYCLE",
    "chronological_age_years",
    "age_topcoded",
    "age_below_20",
    "WTSAF4YR",
    "SDMVSTRA",
    "SDMVPSU",
    "phenotypic_age_years",
    "phenotypic_age_minus_chronological_age_years",
    "formula_variant",
    "formula_intercept",
    "formula_denominator",
    "creatinine_scale",
    "canonical_v1",
    "cross_sectional_reportable",
    "mortality_analysis_authorized",
    "sensitivity_erratum_phenoage_years",
    "sensitivity_creatinine_plus_0_11_phenoage_years",
    "sensitivity_creatinine_plus_0_17_phenoage_years",
    "sensitivity_creatinine_plus_0_23_phenoage_years",
]

canonical_output = canonical.loc[
    :,
    canonical_output_columns,
].copy()

canonical_output.to_parquet(
    CANONICAL_PARTICIPANT_PATH,
    index=False,
)

canonical_output.loc[
    ~canonical_output["age_topcoded"]
].to_parquet(
    NO_TOPCODE_PARTICIPANT_PATH,
    index=False,
)

sensitivity_specs = {
    "canonical_supplement": (
        "phenotypic_age_years"
    ),
    "erratum_constants": (
        "sensitivity_erratum_phenoage_years"
    ),
    "creatinine_plus_0_11_mg_dL": (
        "sensitivity_creatinine_plus_0_11_phenoage_years"
    ),
    "creatinine_plus_0_17_mg_dL": (
        "sensitivity_creatinine_plus_0_17_phenoage_years"
    ),
    "creatinine_plus_0_23_mg_dL": (
        "sensitivity_creatinine_plus_0_23_phenoage_years"
    ),
}

sensitivity_frames = []

for sensitivity_name, value_column in sensitivity_specs.items():
    frame = canonical_output.loc[
        :,
        [
            "SEQN",
            "NHANES_CYCLE",
            "chronological_age_years",
            "age_topcoded",
            "WTSAF4YR",
            "SDMVSTRA",
            "SDMVPSU",
            value_column,
        ],
    ].copy()

    frame = frame.rename(
        columns={
            value_column: "phenotypic_age_years"
        }
    )
    frame["analysis_variant"] = sensitivity_name
    frame["is_canonical"] = (
        sensitivity_name
        == "canonical_supplement"
    )
    frame["delta_from_canonical_years"] = (
        frame["phenotypic_age_years"]
        - canonical_output[
            "phenotypic_age_years"
        ].to_numpy()
    )

    sensitivity_frames.append(frame)

sensitivity_long = pd.concat(
    sensitivity_frames,
    ignore_index=True,
)

sensitivity_long.to_parquet(
    SENSITIVITY_LONG_PATH,
    index=False,
)

sample_flow.to_csv(
    SAMPLE_FLOW_PATH,
    index=False,
)
canonical_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)
regression_checks.to_csv(
    REGRESSION_PATH,
    index=False,
)

sensitivity_summary = (
    sensitivity_long.groupby(
        [
            "NHANES_CYCLE",
            "analysis_variant",
            "is_canonical",
        ],
        observed=True,
    )
    .agg(
        n=("SEQN", "size"),
        mean_delta_from_canonical_years=(
            "delta_from_canonical_years",
            "mean",
        ),
        sd_delta_from_canonical_years=(
            "delta_from_canonical_years",
            lambda values: float(
                np.std(
                    values.to_numpy(dtype=float),
                    ddof=0,
                )
            ),
        ),
        minimum_delta_from_canonical_years=(
            "delta_from_canonical_years",
            "min",
        ),
        maximum_delta_from_canonical_years=(
            "delta_from_canonical_years",
            "max",
        ),
    )
    .reset_index()
)

sensitivity_summary.to_csv(
    SENSITIVITY_SUMMARY_PATH,
    index=False,
)

written_data_paths = [
    CANONICAL_PARTICIPANT_PATH,
    NO_TOPCODE_PARTICIPANT_PATH,
    SENSITIVITY_LONG_PATH,
    SAMPLE_FLOW_PATH,
    SUMMARY_PATH,
    REGRESSION_PATH,
    SENSITIVITY_SUMMARY_PATH,
]

manifest_records = []

for path in written_data_paths:
    manifest_records.append(
        {
            "path": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )

manifest = pd.DataFrame(
    manifest_records
)
manifest.to_csv(
    MANIFEST_PATH,
    index=False,
)

report_summary = canonical_summary.loc[
    canonical_summary["measure"].eq(
        "canonical_supplement"
    ),
    [
        "sample",
        "cycle",
        "n",
        "weighted_population_sum",
        "weighted_mean",
        "taylor_se",
        "ci_low_95",
        "ci_high_95",
    ],
].copy()

report = f"""# AgeLens V1 Canonical Output Rebuild Report

## Status

Canonical rebuild completed successfully.

## Governing Decisions

- D-010: Supplement conversion pair is canonical.
- D-011: age-topcoded participants retained and flagged; no-topcode sensitivity produced.
- D-012: observed modern harmonized creatinine is canonical; three mandatory shifts produced.
- D-013: validation acceptance and BioAge baseline satisfied.
- D-014: exact XPT IBM-zero sentinel normalization was applied upstream.

## Canonical Sample

{report_summary.to_markdown(index=False)}

## Regression Checks

All {len(regression_checks)} canonical regression checks passed.

## Output Scope

The canonical participant file and survey summaries are reportable for cross-sectional AgeLens V1 analyses.

Mortality analysis remains unauthorized and no mortality data were used.

## Required Sensitivities

- Erratum constant pair.
- No-topcode sample.
- Creatinine shifts of +0.11, +0.17, and +0.23 mg/dL.

## Output Manifest

{manifest.to_markdown(index=False)}
"""

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": (
        "08_canonical_output_rebuild.ipynb"
    ),
    "canonical_variant": "supplement",
    "canonical_formula_pair": {
        "intercept": 141.50225,
        "denominator": 0.090165,
    },
    "canonical_creatinine_scale": (
        "observed_modern_harmonized"
    ),
    "required_sensitivities": [
        "erratum_constants",
        "no_topcode",
        "creatinine_plus_0_11_mg_dL",
        "creatinine_plus_0_17_mg_dL",
        "creatinine_plus_0_23_mg_dL",
    ],
    "canonical_participant_rows": int(
        len(canonical_output)
    ),
    "regression_checks_passed": int(
        regression_checks["pass"].sum()
    ),
    "regression_checks_total": int(
        len(regression_checks)
    ),
    "mortality_data_used": False,
    "mortality_analysis_authorized": False,
    "cross_sectional_results_reportable": True,
    "outputs": [
        str(
            path.relative_to(PROJECT_ROOT)
        )
        for path in [
            *written_data_paths,
            MANIFEST_PATH,
            REPORT_PATH,
        ]
    ],
}

METADATA_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Canonical files written:")
for path in [
    *written_data_paths,
    MANIFEST_PATH,
    REPORT_PATH,
    METADATA_PATH,
]:
    print(f"  - {path.relative_to(PROJECT_ROOT)}")


Canonical files written:
  - data\processed\agelens_v1_canonical_complete_case.parquet
  - data\processed\agelens_v1_sensitivity_no_topcode.parquet
  - data\processed\agelens_v1_required_sensitivities_long.parquet
  - results\tables\08_canonical_sample_flow.csv
  - results\tables\08_canonical_survey_summary.csv
  - results\tables\08_canonical_regression_checks.csv
  - results\tables\08_required_sensitivity_summary.csv
  - results\tables\08_canonical_output_manifest.csv
  - docs\methodology\Canonical_V1_Rebuild_Report.md
  - logs\08_canonical_output_rebuild_metadata.json


## 6. Open the cross-sectional release gate

This configuration update occurs only after all canonical outputs and regression checks have been written successfully.

Mortality analysis remains explicitly unauthorized.


In [ ]:
timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

CONFIG_BACKUP_ROOT = (
    LOGS_ROOT
    / "release_backups"
    / timestamp
)
CONFIG_BACKUP_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

CONFIG_BACKUP_PATH = (
    CONFIG_BACKUP_ROOT
    / CONFIG_PATH.name
)
shutil.copy2(
    CONFIG_PATH,
    CONFIG_BACKUP_PATH,
)

config_sha_before = sha256_file(
    CONFIG_PATH
)
config_backup_sha = sha256_file(
    CONFIG_BACKUP_PATH
)

if config_sha_before != config_backup_sha:
    raise RuntimeError(
        "Configuration backup hash does not match."
    )

updated_config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

updated_config["project"]["status"] = (
    "canonical_v1_cross_sectional_ready"
)
updated_config["project"][
    "final_scientific_results_allowed"
] = True

updated_config["release_gates"][
    "canonical_outputs_regenerated"
] = True
updated_config["release_gates"][
    "canonical_regression_checks_passed"
] = True
updated_config["release_gates"][
    "mortality_analysis_authorized"
] = False
updated_config["release_gates"][
    "final_scientific_results_allowed"
] = True
updated_config["release_gates"][
    "cross_sectional_v1_results_allowed"
] = True

updated_config["release_scope"] = {
    "cross_sectional_agelens_v1": "authorized",
    "mortality_analysis": "not_authorized",
    "mortality_data_use": "not_permitted_by_notebook_08",
}

updated_config["canonical_outputs"] = {
    "participant_file": str(
        CANONICAL_PARTICIPANT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "no_topcode_file": str(
        NO_TOPCODE_PARTICIPANT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "required_sensitivities_file": str(
        SENSITIVITY_LONG_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "survey_summary": str(
        SUMMARY_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "manifest": str(
        MANIFEST_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "report": str(
        REPORT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "created_at_utc": metadata[
        "created_at_utc"
    ],
}

CONFIG_PATH.write_text(
    json.dumps(
        updated_config,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

config_sha_after = sha256_file(
    CONFIG_PATH
)

if config_sha_after == config_sha_before:
    raise RuntimeError(
        "The configuration release-gate update did not change "
        "the file."
    )

reloaded_config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

final_gate_checks = pd.DataFrame(
    [
        {
            "check": "Canonical outputs regenerated",
            "pass": reloaded_config[
                "release_gates"
            ][
                "canonical_outputs_regenerated"
            ],
        },
        {
            "check": (
                "Canonical regression checks passed"
            ),
            "pass": reloaded_config[
                "release_gates"
            ][
                "canonical_regression_checks_passed"
            ],
        },
        {
            "check": (
                "Cross-sectional V1 release authorized"
            ),
            "pass": reloaded_config[
                "release_gates"
            ][
                "cross_sectional_v1_results_allowed"
            ],
        },
        {
            "check": (
                "Mortality analysis remains unauthorized"
            ),
            "pass": not reloaded_config[
                "release_gates"
            ][
                "mortality_analysis_authorized"
            ],
        },
        {
            "check": (
                "No Core Evidence Gap remains open"
            ),
            "pass": reloaded_config[
                "governance"
            ][
                "open_core_evidence_gaps"
            ]
            == [],
        },
    ]
)

if not final_gate_checks["pass"].all():
    display(
        final_gate_checks.loc[
            ~final_gate_checks["pass"]
        ]
    )
    raise RuntimeError(
        "Final cross-sectional release-gate verification failed."
    )

FINAL_GATE_PATH = (
    TABLES_ROOT
    / "08_release_gate_checks.csv"
)
final_gate_checks.to_csv(
    FINAL_GATE_PATH,
    index=False,
)

CONFIG_AUDIT_PATH = (
    TABLES_ROOT
    / "08_config_release_update_audit.csv"
)

pd.DataFrame(
    [
        {
            "path": str(
                CONFIG_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "backup": str(
                CONFIG_BACKUP_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256_before": config_sha_before,
            "sha256_backup": config_backup_sha,
            "sha256_after": config_sha_after,
        }
    ]
).to_csv(
    CONFIG_AUDIT_PATH,
    index=False,
)

display(final_gate_checks)

print("✅ Canonical AgeLens V1 rebuild completed.")
print("✅ Cross-sectional V1 results are reportable.")
print("Mortality analysis remains unauthorized.")
print("Mortality data were not used.")
print(f"Configuration backup: {CONFIG_BACKUP_PATH}")


,check,pass
0,Canonical outputs regenerated,True
1,Canonical regression checks passed,True
2,Cross-sectional V1 release authorized,True
3,Mortality analysis remains unauthorized,True
4,No Core Evidence Gap remains open,True


✅ Canonical AgeLens V1 rebuild completed.
✅ Cross-sectional V1 results are reportable.
Mortality analysis remains unauthorized.
Mortality data were not used.
Configuration backup: <PROJECT_ROOT>\logs\release_backups\20260722T161125Z\agelens_config.json
